In [2]:
import numpy as np

# ---- Data from the PDF (inputs and targets) ----
# NOTE: Targets include {1.2, 0.8, 1.8, 0.5}. With a sigmoid output (0..1),
# targets > 1 are not achievable; we keep them as-is to follow the prompt,
# so expect non-zero MSE for those. If your PDF intended a linear output,
# set `linear_output=True` below.
X = np.array([
    [1.0, 1.5],
    [0.2, 0.5],
    [0.75, 0.6],
    [0.6, 0.8]
], dtype=float)

y = np.array([[1.2], [0.8], [1.8], [0.5]], dtype=float)

# ---- Network size ----
n_in, n_hidden, n_out = 2, 2, 1

# ---- Activation functions ----
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def d_sigmoid(a):
    # derivative wrt pre-activation z, given a = sigmoid(z)
    return a * (1.0 - a)

# ---- Forward pass ----
def forward(Xb, W1, b1, W2, b2, linear_output=False):
    z1 = Xb @ W1 + b1  # (batch, 2)
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2  # (batch, 1)
    if linear_output:
        a2 = z2
    else:
        a2 = sigmoid(z2)
    cache = (Xb, z1, a1, z2, a2)
    return a2, cache

# ---- Backward pass (MSE) ----
def backward(yb, cache, W1, b1, W2, b2, linear_output=False):
    Xb, z1, a1, z2, a2 = cache
    m = Xb.shape[0]

    # dL/da2 for MSE = (a2 - y)
    dL_da2 = (a2 - yb)  # (batch, 1)

    # Output layer grad
    if linear_output:
        # a2 = z2, so da2/dz2 = 1
        dL_dz2 = dL_da2
    else:
        dL_dz2 = dL_da2 * d_sigmoid(a2)  # (batch, 1)

    dL_dW2 = (a1.T @ dL_dz2) / m               # (2,1)
    dL_db2 = np.mean(dL_dz2, axis=0, keepdims=True)  # (1,1)

    # Hidden layer grad
    dL_da1 = dL_dz2 @ W2.T                    # (batch,2)
    dL_dz1 = dL_da1 * d_sigmoid(a1)           # (batch,2)

    dL_dW1 = (Xb.T @ dL_dz1) / m              # (2,2)
    dL_db1 = np.mean(dL_dz1, axis=0, keepdims=True)  # (1,2)

    return dL_dW1, dL_db1, dL_dW2, dL_db2

# ---- Training helpers ----
def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))

def apply_updates(W1, b1, W2, b2, grads, lr):
    dW1, db1, dW2, db2 = grads
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2
    return W1, b1, W2, b2

# ---- Training modes ----
def train_stochastic(X, y, W1, b1, W2, b2, lr=0.5, epochs=5, linear_output=False, tol=1e-8):
    n = len(X)
    prev = None
    for ep in range(1, epochs+1):
        idx = np.random.permutation(n)
        for i in idx:
            Xb = X[i:i+1]
            yb = y[i:i+1]
            yhat, cache = forward(Xb, W1, b1, W2, b2, linear_output=linear_output)
            grads = backward(yb, cache, W1, b1, W2, b2, linear_output=linear_output)
            W1, b1, W2, b2 = apply_updates(W1, b1, W2, b2, grads, lr)
        # epoch MSE
        yhat_all, _ = forward(X, W1, b1, W2, b2, linear_output=linear_output)
        cur = mse(y, yhat_all)
        print(f"[Stochastic] Epoch {ep} MSE = {cur:.6f}")
        if prev is not None and abs(prev - cur) < tol:
            break
        prev = cur
    return W1, b1, W2, b2, cur

def train_full_batch(X, y, W1, b1, W2, b2, lr=0.5, epochs=5, linear_output=False, tol=1e-8):
    prev = None
    for ep in range(1, epochs+1):
        yhat, cache = forward(X, W1, b1, W2, b2, linear_output=linear_output)
        grads = backward(y, cache, W1, b1, W2, b2, linear_output=linear_output)
        W1, b1, W2, b2 = apply_updates(W1, b1, W2, b2, grads, lr)
        cur = mse(y, yhat)
        print(f"[Full-Batch] Epoch {ep} MSE = {cur:.6f}")
        if prev is not None and abs(prev - cur) < tol:
            break
        prev = cur
    return W1, b1, W2, b2, cur

def train_mini_batch(X, y, W1, b1, W2, b2, lr=0.5, epochs=5, batch_size=2, linear_output=False, tol=1e-8):
    n = len(X)
    prev = None
    for ep in range(1, epochs+1):
        idx = np.random.permutation(n)
        for k in range(0, n, batch_size):
            sel = idx[k:k+batch_size]
            Xb = X[sel]
            yb = y[sel]
            yhat, cache = forward(Xb, W1, b1, W2, b2, linear_output=linear_output)
            grads = backward(yb, cache, W1, b1, W2, b2, linear_output=linear_output)
            W1, b1, W2, b2 = apply_updates(W1, b1, W2, b2, grads, lr)
        # epoch MSE
        yhat_all, _ = forward(X, W1, b1, W2, b2, linear_output=linear_output)
        cur = mse(y, yhat_all)
        print(f"[Mini-Batch] Epoch {ep} MSE = {cur:.6f}")
        if prev is not None and abs(prev - cur) < tol:
            break
        prev = cur
    return W1, b1, W2, b2, cur

# ---- Run all three modes ----
def run_part_a():
    # === Paste the EXACT initial weights from your PDF here ===
    # Hidden-from-Input weights (shape 2x2) and biases (shape 1x2):
    W1_init = np.array([[0.10, -0.20],
                        [0.30,  0.40]], dtype=float)    # <-- REPLACE with your PDF values
    b1_init = np.array([[0.00, 0.00]], dtype=float)     # <-- REPLACE if your PDF includes hidden biases

    # Output-from-Hidden weights (shape 2x1) and bias (shape 1x1):
    W2_init = np.array([[0.25],
                        [-0.15]], dtype=float)          # <-- REPLACE with your PDF values
    b2_init = np.array([[0.00]], dtype=float)           # <-- REPLACE if your PDF includes output bias

    # Copy for each mode
    W1s, b1s, W2s, b2s = map(np.copy, (W1_init, b1_init, W2_init, b2_init))
    W1b, b1b, W2b, b2b = map(np.copy, (W1_init, b1_init, W2_init, b2_init))
    W1m, b1m, W2m, b2m = map(np.copy, (W1_init, b1_init, W2_init, b2_init))

    lr = 0.5
    epochs = 5
    # If your PDF truly requires sigmoid at output, leave as False.
    # If it intended linear output neuron, set linear_output=True.
    linear_output = False

    print("\n--- PART A: Stochastic ---")
    W1s, b1s, W2s, b2s, mse_s = train_stochastic(X, y, W1s, b1s, W2s, b2s, lr=lr, epochs=epochs, linear_output=linear_output)

    print("\n--- PART A: Full Batch ---")
    W1b, b1b, W2b, b2b, mse_b = train_full_batch(X, y, W1b, b1b, W2b, b2b, lr=lr, epochs=epochs, linear_output=linear_output)

    print("\n--- PART A: Mini-Batch (size=2) ---")
    W1m, b1m, W2m, b2m, mse_m = train_mini_batch(X, y, W1m, b1m, W2m, b2m, lr=lr, epochs=epochs, batch_size=2, linear_output=linear_output)

    print("\nFinal MSEs -> Stochastic: {:.6f}, Full-Batch: {:.6f}, Mini-Batch: {:.6f}".format(mse_s, mse_b, mse_m))

if __name__ == "__main__":
    # Comment this out if you're pasting PART B below in the same file
    run_part_a()



--- PART A: Stochastic ---
[Stochastic] Epoch 1 MSE = 0.445876
[Stochastic] Epoch 2 MSE = 0.381329
[Stochastic] Epoch 3 MSE = 0.344793
[Stochastic] Epoch 4 MSE = 0.322424
[Stochastic] Epoch 5 MSE = 0.306778

--- PART A: Full Batch ---
[Full-Batch] Epoch 1 MSE = 0.549531
[Full-Batch] Epoch 2 MSE = 0.518205
[Full-Batch] Epoch 3 MSE = 0.490427
[Full-Batch] Epoch 4 MSE = 0.465992
[Full-Batch] Epoch 5 MSE = 0.444609

--- PART A: Mini-Batch (size=2) ---
[Mini-Batch] Epoch 1 MSE = 0.490638
[Mini-Batch] Epoch 2 MSE = 0.444591
[Mini-Batch] Epoch 3 MSE = 0.409973
[Mini-Batch] Epoch 4 MSE = 0.383137
[Mini-Batch] Epoch 5 MSE = 0.362501

Final MSEs -> Stochastic: 0.306778, Full-Batch: 0.444609, Mini-Batch: 0.362501


In [5]:
# =========================
# EXERCISE 5 — PART B.1
# Breast Cancer — Binary classification (PyTorch)
# Preprocess: StandardScaler
# Model: 1 hidden (ReLU), sigmoid output
# Loss: BCELoss
# =========================

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load + preprocess
data = load_breast_cancer()
X = data.data
y = data.target.reshape(-1, 1)  # binary labels to shape (N,1)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)

# Model
class BCNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 16)
        self.fc2 = nn.Linear(16, 1)
        self.act = nn.ReLU()
        self.out_act = nn.Sigmoid()
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.out_act(self.fc2(x))
        return x

model = BCNet(X_train.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train
model.train()
for epoch in range(100):
    optimizer.zero_grad()
    preds = model(X_train)
    loss = criterion(preds, y_train)
    loss.backward()
    optimizer.step()

# Accuracies
model.eval()
with torch.no_grad():
    train_pred = (model(X_train) >= 0.5).float()
    test_pred  = (model(X_test)  >= 0.5).float()
    train_acc = accuracy_score(y_train.numpy(), train_pred.numpy())
    test_acc  = accuracy_score(y_test.numpy(),  test_pred.numpy())

print(f"[Breast Cancer] Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")


[Breast Cancer] Train Acc: 0.9538 | Test Acc: 0.9737


In [4]:
# =========================
# EXERCISE 5 — PART B.2
# Wine — Multiclass classification (PyTorch)
# Preprocess: StandardScaler
# Model: 2 hidden layers (ReLU), raw logits
# Loss: CrossEntropyLoss
# =========================

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load + preprocess
wine = load_wine()
X = wine.data
y = wine.target  # labels 0..2

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.long)

# Model
class WineNet(nn.Module):
    def __init__(self, in_dim, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, num_classes)
        self.act = nn.ReLU()
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.fc3(x)  # raw logits
        return x

model = WineNet(X_train.shape[1], 3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train
model.train()
for epoch in range(200):
    optimizer.zero_grad()
    logits = model(X_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()

# Evaluate
model.eval()
with torch.no_grad():
    tr_logits = model(X_train)
    te_logits = model(X_test)
    tr_pred = tr_logits.argmax(dim=1)
    te_pred = te_logits.argmax(dim=1)
    tr_acc = accuracy_score(y_train.numpy(), tr_pred.numpy())
    te_acc = accuracy_score(y_test.numpy(),  te_pred.numpy())

print(f"[Wine] Train Acc: {tr_acc:.4f} | Test Acc: {te_acc:.4f}")


[Wine] Train Acc: 1.0000 | Test Acc: 0.9722


In [6]:
# =========================
# EXERCISE 5 — PART B.3
# Digits — 10-class classification (PyTorch)
# Preprocess: flatten 8x8 -> 64, StandardScaler
# Model: 1 hidden (ReLU), raw logits
# Loss: CrossEntropyLoss
# =========================

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load + preprocess
digits = load_digits()
X = digits.images.reshape(len(digits.images), -1)  # (N, 64)
y = digits.target                                   # labels 0..9

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.long)

# Model
class DigitsNet(nn.Module):
    def __init__(self, in_dim=64, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 64)
        self.fc2 = nn.Linear(64, num_classes)
        self.act = nn.ReLU()
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.fc2(x)  # logits
        return x

model = DigitsNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train
model.train()
for epoch in range(50):
    optimizer.zero_grad()
    logits = model(X_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()

# Evaluate
model.eval()
with torch.no_grad():
    tr_logits = model(X_train)
    te_logits = model(X_test)
    tr_pred = tr_logits.argmax(dim=1)
    te_pred = te_logits.argmax(dim=1)
    tr_acc = accuracy_score(y_train.numpy(), tr_pred.numpy())
    te_acc = accuracy_score(y_test.numpy(),  te_pred.numpy())

print(f"[Digits] Train Acc: {tr_acc:.4f} | Test Acc: {te_acc:.4f}")

# Show a few predictions (first 5 test samples)
with torch.no_grad():
    sample_logits = model(X_test[:5])
    sample_pred = sample_logits.argmax(dim=1).numpy()
print("First 5 test labels:     ", y_test[:5].numpy())
print("First 5 predicted labels:", sample_pred)


[Digits] Train Acc: 0.8594 | Test Acc: 0.8222
First 5 test labels:      [5 2 8 1 7]
First 5 predicted labels: [9 2 1 1 7]
